# Neural Network Models

In [ ]:
!pip install scikeras

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import RandomizedSearchCV
from tensorflow.keras.layers import Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

### **Initial MLP**


In [ ]:
# Load dataset
df = pd.read_excel('TCI_sas (1).xlsx', sheet_name="Sheet1", usecols=["Hrd", "milkperiod", "zdate", "zdate_month", "firstmilk", "firstmilkdays",
                                                                               "prelendays", "drylendays", "milkdays",
                                                                               "previous_Milk305",
                                                                               "firstmilk_previous", "SCS_305"])

In [ ]:
df['drylendays'] = df['drylendays'].fillna(df['drylendays'].mean())
df['milkdays'] = df['milkdays'].fillna(df['milkdays'].mean())

# Separate features and target
X = df.drop(columns=["firstmilk"])
y = df["firstmilk"].astype(float)

# First split: separate test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Second split: separate train (70%) and validation (15%) from remaining data
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)

numerical_columns = ["milkperiod", "zdate", "zdate_month", "firstmilkdays", "prelendays", "drylendays", "milkdays",
                     "previous_Milk305", "firstmilk_previous", "SCS_305"]

scaler = StandardScaler()
scaler.fit(X_train[numerical_columns])
X_train[numerical_columns] = scaler.transform(X_train[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

In [ ]:
### **Define functions for additional metrics**
def calculate_mpe(y_true, y_pred):
    mask = y_true != 0
    return np.mean((y_true[mask] - y_pred[mask]) / y_true[mask] * 100)

def calculate_smape(y_true, y_pred):
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0
    return np.mean(numerator[mask] / denominator[mask] * 100)

def calculate_sdr(y_true, y_pred):
    return np.std(y_pred) / np.std(y_true)

In [ ]:
print("\n=== MLP Neural Network Model ===")
# Initialize MLP model with simple architecture
mlp_model = MLPRegressor(hidden_layer_sizes=(100,), activation='relu', max_iter=500, random_state=42, learning_rate_init=0.001, verbose=3)
mlp_model.fit(X_train[numerical_columns], y_train)

# Predict on test set
y_test_pred_mlp = mlp_model.predict(X_test[numerical_columns])

# Calculate metrics
test_r2_mlp = r2_score(y_test, y_test_pred_mlp)
test_mae_mlp = mean_absolute_error(y_test, y_test_pred_mlp)
test_mse_mlp = mean_squared_error(y_test, y_test_pred_mlp)
test_rmse_mlp = np.sqrt(test_mse_mlp)
test_mpe_mlp = calculate_mpe(y_test, y_test_pred_mlp)
test_smape_mlp = calculate_smape(y_test, y_test_pred_mlp)
test_sdr_mlp = calculate_sdr(y_test, y_test_pred_mlp)

# Print results
print("Test Set (MLP):")
print(f"R²    : {test_r2_mlp:.4f}")
print(f"MAE   : {test_mae_mlp:.4f}")
print(f"RMSE  : {test_rmse_mlp:.4f}")
print(f"MPE   : {test_mpe_mlp:.4f}")
print(f"sMAPE : {test_smape_mlp:.4f}")
print(f"SDR   : {test_sdr_mlp:.4f}")

### **Initial Deep Neural Network**


In [ ]:
# Load dataset
df = pd.read_excel('TCI_sas (1).xlsx', sheet_name="Sheet1", usecols=["Hrd", "milkperiod", "zdate", "zdate_month", "firstmilk", "firstmilkdays",
                                                                               "prelendays", "drylendays", "milkdays",
                                                                               "previous_Milk305",
                                                                               "firstmilk_previous", "SCS_305"])

In [ ]:
df['drylendays'] = df['drylendays'].fillna(df['drylendays'].mean())
df['milkdays'] = df['milkdays'].fillna(df['milkdays'].mean())

# Separate features and target
X = df.drop(columns=["firstmilk"])
y = df["firstmilk"].astype(float)

# First split: separate test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Second split: separate train (70%) and validation (15%) from remaining data
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)

numerical_columns = ["milkperiod", "zdate", "zdate_month", "firstmilkdays", "prelendays", "drylendays", "milkdays",
                     "previous_Milk305", "firstmilk_previous", "SCS_305"]

scaler = StandardScaler()
scaler.fit(X_train[numerical_columns])
X_train[numerical_columns] = scaler.transform(X_train[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

In [ ]:
print("\n=== Deep Neural Network Model ===")
# Define a simple DNN model
dnn_model = Sequential([
    Dense(64, activation='relu', input_shape=(len(numerical_columns),)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='linear')
])

# Compile the model
dnn_model.compile(optimizer='adam', loss='mse')

# Train the model
dnn_model.fit(X_train[numerical_columns], y_train, validation_data=(X_val[numerical_columns], y_val), epochs=50, batch_size=32, verbose=3)

# Predict on test set
y_test_pred_dnn = dnn_model.predict(X_test[numerical_columns]).flatten()

# Calculate metrics
test_r2_dnn = r2_score(y_test, y_test_pred_dnn)
test_mae_dnn = mean_absolute_error(y_test, y_test_pred_dnn)
test_mse_dnn = mean_squared_error(y_test, y_test_pred_dnn)
test_rmse_dnn = np.sqrt(test_mse_dnn)
test_mpe_dnn = calculate_mpe(y_test, y_test_pred_dnn)
test_smape_dnn = calculate_smape(y_test, y_test_pred_dnn)
test_sdr_dnn = calculate_sdr(y_test, y_test_pred_dnn)

# Print results
print("Test Set (DNN):")
print(f"R²    : {test_r2_dnn:.4f}")
print(f"MAE   : {test_mae_dnn:.4f}")
print(f"RMSE  : {test_rmse_dnn:.4f}")
print(f"MPE   : {test_mpe_dnn:.4f}")
print(f"sMAPE : {test_smape_dnn:.4f}")
print(f"SDR   : {test_sdr_dnn:.4f}")

## **Print results into a new CSV file**


In [ ]:
# Function to print existing CSV content
def print_csv_content(file_path):
    try:
        existing_df = pd.read_csv(file_path, index_col=0)
        return existing_df
    except FileNotFoundError:
        print("\nNo existing test evaluation table found.")
        return pd.DataFrame()

# Load existing CSV content from new file (if exists)
new_file_path = 'test_evaluation_metrics1.csv'
existing_df = print_csv_content(new_file_path)

# Save results to a new CSV file
mlp_results = {
    'MLP': {
        'Test_R2': test_r2_mlp, 'Test_MAE': test_mae_mlp, 'Test_RMSE': test_rmse_mlp,
        'Test_MPE': test_mpe_mlp, 'Test_sMAPE': test_smape_mlp, 'Test_SDR': test_sdr_mlp
    }
}
mlp_df = pd.DataFrame.from_dict(mlp_results, orient='index')

dnn_results = {
    'DNN': {
        'Test_R2': test_r2_dnn, 'Test_MAE': test_mae_dnn, 'Test_RMSE': test_rmse_dnn,
        'Test_MPE': test_mpe_dnn, 'Test_sMAPE': test_smape_dnn, 'Test_SDR': test_sdr_dnn
    }
}
dnn_df = pd.DataFrame.from_dict(dnn_results, orient='index')

# Combine new results
new_results_df = pd.concat([mlp_df, dnn_df])

# If existing data exists, append it; otherwise, start with new results
if not existing_df.empty:
    comparison_df = pd.concat([new_results_df, existing_df])
else:
    comparison_df = new_results_df

# Save to the new file
comparison_df.to_csv(new_file_path, index=True)
print("\n=== Updated Test Evaluation Metrics Saved to test_evaluation_metrics.csv ===")
print(comparison_df)